# Tutorial: GraphTrust-SWE synthetic CPU demonstration

This notebook is for researchers evaluating repository-graph evidence. It needs Python 3.11+ and the local GraphTrust-SWE checkout; no GPU, weights, repository download, or network is required. Run `python -m pip install -e .` once from the repository root.

The learning goals are to generate six arms with identical source, inspect evaluator isolation, and reproduce scored rows and paired summaries. **All outputs here are synthetic wiring checks, never paper evidence.**

1. Locate the checkout and run the CLI demo.
2. Verify source and prompt invariants.
3. Inspect failure-preserving reports.
4. Try a reference-file coverage exercise.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
repo = Path(os.environ['GRAPHTRUST_REPO']).resolve() if os.environ.get('GRAPHTRUST_REPO') else next((p for p in candidates if (p / 'configs' / 'protocol.json').is_file() and (p / 'fixtures' / 'synthetic').is_dir()), None)
assert repo is not None, 'Run from the checkout, or set GRAPHTRUST_REPO to its absolute path.'
out = Path(os.environ['GRAPHTRUST_DEMO_OUT']).resolve() if os.environ.get('GRAPHTRUST_DEMO_OUT') else repo / 'runs' / 'demo'
print('Checkout:', repo)


## Generate the resource

The CLI builds a conservative graph from the bundled fixtures, constructs all six conditions, and runs a clearly labeled mock backend. Re-running resumes recorded logical calls. A mock outcome says nothing about Gemma reliability.


In [ ]:
command = [sys.executable, '-m', 'graphtrust', 'demo', '--out', str(out)]
result = subprocess.run(command, cwd=repo, capture_output=True, text=True)
if result.returncode:
    print(result.stderr)
result.check_returncode()
print(result.stdout)


## Verify identical source exposure and evaluator isolation

A source hash is only useful if it matches the actual source section. Check the bytes inside each prompt as well as the metadata. Corruption labels may be stored beside prompts for evaluation, but the prompt itself must never reveal them.


In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

contexts = read_jsonl(out / 'contexts.jsonl')
conditions = {row['condition'] for row in contexts}
assert conditions == {'source_only', 'clean', 'deleted', 'rewired', 'clean_checked', 'rewired_checked'}
source_sections = [row['prompt'].split('SOURCE PACK\n', 1)[1].split('END SOURCE PACK\n', 1)[0] for row in contexts]
assert len(set(source_sections)) == 1
assert len({row['source_sha256'] for row in contexts}) == 1
for row, source in zip(contexts, source_sections):
    assert hashlib.sha256(source.encode('utf-8')).hexdigest() == row['source_sha256']
    assert hashlib.sha256(row['prompt'].encode('utf-8')).hexdigest() == row['context_sha256']
    assert all(marker not in row['prompt'] for marker in ['known_false_edge_ids', 'test_patch', '\"patch\":'])
print('Verified six arms and source invariance.')


## Read an auditable report

The report retains every attempt. Seeds are averaged within an issue, and models are analyzed separately. This demo has one synthetic issue, so an uncertainty interval across repositories is unavailable. Optional matplotlib figures, when installed, are visibly marked synthetic.


In [ ]:
summary = json.loads((out / 'report' / 'summary.json').read_text(encoding='utf-8'))
assert summary['synthetic']
for model, details in summary['models'].items():
    rows = [(arm, values['runs'], values['issues'], values['metrics']['file_recall_at5']['mean']) for arm, values in details['conditions'].items()]
    print(model, '\ncondition, runs, issues, recall@5:', rows)
print('Artifacts:', sorted(p.name for p in (out / 'report').iterdir()))


## Exercise: candidate coverage

What happens to localization recall if the evaluator's reference edit includes a file absent from the source pack? Add `unavailable.py` to the synthetic reference list. The case must remain in the denominator; the coverage ceiling should fall. This is a demonstration of scoring policy, not a model experiment.


In [ ]:
from graphtrust.metrics import evaluate_runs, summarize
records = read_jsonl(out / 'records.jsonl')
case_id = contexts[0]['case_id']
exercise_gold = {case_id: ['normalize.py', 'app.py', 'unavailable.py']}
scored = evaluate_runs(records, exercise_gold, contexts)
assert all(row['gold_file_count'] == 3 for row in scored)
assert all(row['coverage_ceiling'] <= 2 / 3 for row in scored)
print('Candidate coverage ceiling:', scored[0]['coverage_ceiling'])


## Pitfall and extension

Do not count twelve mock calls as twelve independent issues: this notebook still has one issue and one repository. Inspect `results.csv` to trace a summary back to its attempt. For a real study, run the preregistered development pilot with a pinned model and exact token counts; retain failures and do not use holdout outcomes to tune retrieval.
